In [ ]:
import json
import numpy as np
from collections import Counter

# Load the annotations file
json_path = '/fs/ess/PAS2136/ggr_data/results/GGR2020_subset_refactor_final/ia_classifier/ia_annots.json'
with open(json_path, 'r') as f:
    data = json.load(f)

# Filter for Grevy's zebra only (category_id = 0)
grevys_annotations = [ann for ann in data['annotations'] if ann['category_id'] == 0]

# Define CA_score groups
ca_groups = {
    'NaN': [],
    '< 0.35': [],
    '≥ 0.35 and < 0.65': [],
    '≥ 0.65 and < 0.9': [],
    '≥ 0.9': []
}

# Categorize annotations by CA_score
for ann in grevys_annotations:
    if 'CA_score' not in ann or ann['CA_score'] is None or np.isnan(ann['CA_score']):
        ca_groups['NaN'].append(ann)
    else:
        score = ann['CA_score']
        if score < 0.35:
            ca_groups['< 0.35'].append(ann)
        elif score < 0.65:
            ca_groups['≥ 0.35 and < 0.65'].append(ann)
        elif score < 0.9:
            ca_groups['≥ 0.65 and < 0.9'].append(ann)
        else:
            ca_groups['≥ 0.9'].append(ann)

# Print detailed statistics per group
print("="*80)
print("DETAILED STATISTICS PER CA_SCORE GROUP")
print("="*80)

for group_name, group_anns in ca_groups.items():
    if not group_anns:
        continue
        
    print(f"\n{group_name} Group:")
    print(f"  Total annotations: {len(group_anns)}")
    print(f"  Percentage of total: {len(group_anns)/len(grevys_annotations)*100:.1f}%")
    
    # For non-NaN groups, calculate CA_score statistics
    if group_name != 'NaN':
        ca_scores = [ann['CA_score'] for ann in group_anns]
        print(f"  CA_score range: [{min(ca_scores):.3f}, {max(ca_scores):.3f}]")
        print(f"  CA_score mean: {np.mean(ca_scores):.3f}")
        print(f"  CA_score std: {np.std(ca_scores):.3f}")
    
    # Analyze viewpoints if available
    if any('viewpoint' in ann for ann in group_anns):
        viewpoints = [ann.get('viewpoint', 'unknown') for ann in group_anns]
        viewpoint_counts = Counter(viewpoints)
        print(f"  Viewpoint distribution:")
        for vp, count in sorted(viewpoint_counts.items()):
            print(f"    {vp}: {count} ({count/len(group_anns)*100:.1f}%)")
    
    # Analyze IA consensus if available
    if any('annotations_census' in ann for ann in group_anns):
        ia_values = [ann.get('annotations_census', 'unknown') for ann in group_anns]
        ia_counts = Counter(ia_values)
        print(f"  IA consensus distribution:")
        for ia, count in sorted(ia_counts.items()):
            print(f"    {ia}: {count} ({count/len(group_anns)*100:.1f}%)")
    
    # Sample annotation UUIDs
    print(f"  Sample annotation UUIDs (first 5):")
    for i, ann in enumerate(group_anns[:5]):
        print(f"    {ann['uuid']}")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Total Grevy's zebra annotations: {len(grevys_annotations)}")
print(f"Annotations with valid CA_score: {len(grevys_annotations) - len(ca_groups['NaN'])}")
print(f"Annotations with NaN CA_score: {len(ca_groups['NaN'])}")

In [ ]:
import json
import numpy as np

# Load the annotations file
json_path = '/fs/ess/PAS2136/ggr_data/results/GGR2020_subset_refactor_final/ia_classifier/ia_annots.json'
with open(json_path, 'r') as f:
    data = json.load(f)

# Count total annotations
total_annotations = len(data['annotations'])
print(f"Total annotations: {total_annotations}")

# Build category mapping
cat_id_to_name = {cat['id']: cat['species'] for cat in data['categories']}
print(f"Categories: {cat_id_to_name}")

# Filter for Grevy's zebra only (category_id = 0)
grevys_annotations = [ann for ann in data['annotations'] if ann['category_id'] == 0]
print(f"\nGrevy's zebra annotations: {len(grevys_annotations)}")

# Count annotations per CA_score group
ca_groups_count = {
    'NaN': 0,
    '< 0.35': 0,
    '≥ 0.35 and < 0.65': 0,
    '≥ 0.65 and < 0.9': 0,
    '≥ 0.9': 0
}

# Also collect scores for statistics
ca_scores = []

for ann in grevys_annotations:
    if 'CA_score' not in ann or ann['CA_score'] is None or np.isnan(ann['CA_score']):
        ca_groups_count['NaN'] += 1
    else:
        score = ann['CA_score']
        ca_scores.append(score)
        if score < 0.35:
            ca_groups_count['< 0.35'] += 1
        elif score < 0.65:
            ca_groups_count['≥ 0.35 and < 0.65'] += 1
        elif score < 0.9:
            ca_groups_count['≥ 0.65 and < 0.9'] += 1
        else:
            ca_groups_count['≥ 0.9'] += 1

# Display counts per CA_score group
print("\nAnnotations per CA_score group:")
for group_name, count in ca_groups_count.items():
    percentage = (count / len(grevys_annotations)) * 100 if len(grevys_annotations) > 0 else 0
    print(f"{group_name}: {count} ({percentage:.1f}%)")

# Calculate statistics for non-NaN CA_scores
if ca_scores:
    print(f"\nCA_score statistics (excluding NaN):")
    print(f"  Count: {len(ca_scores)}")
    print(f"  Mean: {np.mean(ca_scores):.3f}")
    print(f"  Std: {np.std(ca_scores):.3f}")
    print(f"  Min: {np.min(ca_scores):.3f}")
    print(f"  Max: {np.max(ca_scores):.3f}")
    print(f"  Q1: {np.percentile(ca_scores, 25):.3f}")
    print(f"  Median: {np.median(ca_scores):.3f}")
    print(f"  Q3: {np.percentile(ca_scores, 75):.3f}")

In [ ]:
import json
import random
import matplotlib.pyplot as plt
from PIL import Image
import os
import math
import numpy as np

# Parameters
json_path = '/fs/ess/PAS2136/ggr_data/results/GGR2020_subset_refactor_final/ia_classifier/ia_annots.json'
n_per_group = 9  # Total crops per CA_score group (max 9, arranged 3x3)
max_per_row = 3  # No more than 3 crops per row

# Load the JSON file
with open(json_path, 'r') as f:
    data = json.load(f)

# Build image mapping (using uuid as key)
image_map = {img['uuid']: img['image_path'] for img in data['images']}

# Filter for Grevy's zebra only (category_id = 0)
grevys_annotations = [ann for ann in data['annotations'] if ann['category_id'] == 0]
print(f"Total Grevy's zebra annotations: {len(grevys_annotations)}")

# Define CA_score groups
ca_groups = {
    'NaN': [],
    '< 0.35': [],
    '≥ 0.35 and < 0.65': [],
    '≥ 0.65 and < 0.9': [],
    '≥ 0.9': []
}

# Categorize annotations by CA_score
for ann in grevys_annotations:
    if 'CA_score' not in ann or ann['CA_score'] is None or np.isnan(ann['CA_score']):
        ca_groups['NaN'].append(ann)
    else:
        score = ann['CA_score']
        if score < 0.35:
            ca_groups['< 0.35'].append(ann)
        elif score < 0.65:
            ca_groups['≥ 0.35 and < 0.65'].append(ann)
        elif score < 0.9:
            ca_groups['≥ 0.65 and < 0.9'].append(ann)
        else:
            ca_groups['≥ 0.9'].append(ann)

# For each CA_score group, select random annotation crops
for group_name, group_anns in ca_groups.items():
    if not group_anns:
        print(f"No annotations found for CA_score group '{group_name}'")
        continue
        
    selected_anns = random.sample(group_anns, min(n_per_group, len(group_anns)))
    n = len(selected_anns)
    n_rows = math.ceil(n / max_per_row)
    print(f"\nCA_score group '{group_name}': showing {n} annotation crops from {len(group_anns)} total")

    fig, axes = plt.subplots(n_rows, max_per_row, figsize=(4*max_per_row, 4*n_rows))
    
    # Handle single subplot case
    if n_rows == 1 and max_per_row == 1:
        axes = [[axes]]
    elif n_rows == 1:
        axes = [axes]
    elif max_per_row == 1:
        axes = [[ax] for ax in axes]
    
    # Flatten for easy indexing
    axes_flat = [ax for row in axes for ax in row]

    for idx, ann in enumerate(selected_anns):
        img_uuid = ann['image_uuid']
        img_path = image_map.get(img_uuid)
        
        if not img_path:
            print(f"Warning: No image found for image_uuid {img_uuid}")
            axes_flat[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=axes_flat[idx].transAxes)
            axes_flat[idx].axis('off')
            continue
            
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}")
            axes_flat[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=axes_flat[idx].transAxes)
            axes_flat[idx].axis('off')
            continue

        try:
            img = Image.open(img_path)
            x, y, w, h = map(int, ann['bbox'])
            # Ensure coordinates are within bounds
            x = max(0, x)
            y = max(0, y)
            x2 = min(img.width, x + w)
            y2 = min(img.height, y + h)
            
            crop = img.crop((x, y, x2, y2))
            axes_flat[idx].imshow(crop)
            
            # Format title based on group
            if group_name == 'NaN':
                title = f"CA: NaN\nUUID: {ann['uuid'][:8]}"
            else:
                ca_score = ann.get('CA_score', 'N/A')
                title = f"CA: {ca_score:.3f}\nUUID: {ann['uuid'][:8]}"
            
            axes_flat[idx].set_title(title, fontsize=10)
            axes_flat[idx].axis('off')

            # Print details
            print(f"  Ann UUID: {ann['uuid']}, Image: {os.path.basename(img_path)}, CA_score: {ann.get('CA_score', 'NaN')}, BBox: {ann['bbox']}")
            
        except Exception as e:
            print(f"Error processing annotation {ann['uuid']}: {e}")
            axes_flat[idx].text(0.5, 0.5, 'Error loading image', ha='center', va='center', transform=axes_flat[idx].transAxes)
            axes_flat[idx].axis('off')

    # Hide any unused axes
    for j in range(idx+1, n_rows*max_per_row):
        if j < len(axes_flat):
            axes_flat[j].axis('off')

    plt.suptitle(f"CA_score Group: {group_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()